In [8]:
import os
import time
import random
import warnings

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as F

from tqdm import tqdm
from torch.export import export
from torchvision import datasets
from torchvision.io import decode_image
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e

from executorch import version
from executorch.runtime import Runtime
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config


random.seed(0)
warnings.simplefilter("ignore", FutureWarning)
print("Executorch version: ", version.__version__)

Executorch version:  1.1.0


In [9]:
model_constructors = [
    models.efficientnet_v2_s,
]

model_weights = [
    models.EfficientNet_V2_S_Weights.DEFAULT,
]

model_resize_sizes = [
    [384, 384],
]

In [10]:
# Directory to store the exported .pte files
EXPORT_DIR = "exports"

# Suffix for models
SUFFIX = "_xnnpack_int8"

# Paths to the exported .pte files
MODEL_PATH: dict[str, str] = {}

def preprocess(img: torch.Tensor, resize_size: list[int]) -> torch.Tensor:
    img = F.resize(img, resize_size, interpolation=F.InterpolationMode.BILINEAR, antialias=True)
    img = F.convert_image_dtype(img, torch.float)
    return img

In [11]:
print("Exporting models...")


class Model(nn.Module):
    def __init__(self, constructor, weights):
        super().__init__()
        self.mean = torch.as_tensor(weights.transforms().mean).view(-1, 1, 1)
        self.std = torch.as_tensor(weights.transforms().std).view(-1, 1, 1)
        self.model = constructor(weights=weights).eval()

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x)


# We use a common dataset of images for calibration of activation ranges. This
# particular dataset is simply reused from the semantic segmentation export
# script, but any dataset of images would work. We just need to make sure to
# apply the same preprocessing (resizing and normalization) to the images before
# passing them to the model for calibration.
#
# NOTE: There is some problem with getting the VOCSegmentation dataset to
# download in torchvision (https://github.com/pytorch/vision/issues/9269). You
# can download it from a mirror using the following commands:
# ```
# mkdir -p datasets/VOCSegmentation
# cd datasets/VOCSegmentation
# curl https://dataset.bj.bcebos.com/voc/VOCtrainval_11-May-2012.tar
# ```
#
calibration_dataset = datasets.VOCSegmentation(root="datasets/VOCSegmentation", year="2012", image_set="train")
calibration_size = 100
calibration_imgs = random.sample(calibration_dataset.images, k=calibration_size)

for constructor, weights, resize_size in zip(model_constructors, model_weights, model_resize_sizes):
    model = Model(constructor, weights)
    inputs = (torch.randn(1, 3, *resize_size),)

    # NOTE: Quantization pipeline is based on the ExecuTorch example for xnnpack
    # backend:
    # https://docs.pytorch.org/executorch/1.1/backends/xnnpack/xnnpack-quantization.html

    # Prepare quantizer
    # -----------------
    quantizer = XNNPACKQuantizer()
    quantizer.set_global(get_symmetric_quantization_config(is_per_channel=True))

    # Prepare model for PTQ
    # ---------------------
    training_ep = export(model, inputs).module()
    prepared_model = prepare_pt2e(training_ep, quantizer)

    # Calibrate activation ranges
    # ---------------------------
    for img in tqdm(calibration_imgs, desc="Calibrating"):
        img = decode_image(img).unsqueeze(0)
        img = preprocess(img, resize_size)
        prepared_model(img)

    # Export quantized model
    # ----------------------
    quantized_model = convert_pt2e(prepared_model)

    program = to_edge_transform_and_lower(export(quantized_model, inputs), partitioner=[XnnpackPartitioner()])
    program = program.to_executorch()

    with open(path := f"{EXPORT_DIR}/{constructor.__name__}{SUFFIX}.pte", "wb") as f:
        MODEL_PATH[constructor.__name__] = path
        f.write(program.buffer)

    print(f"Exported {path}\n")

Exporting models...


Calibrating: 100%|██████████| 100/100 [02:20<00:00,  1.41s/it]


Exported exports/efficientnet_v2_s_xnnpack_int8.pte



In [12]:
img = decode_image("assets/dog.png")

for constructor, weights, resize_size in zip(model_constructors, model_weights, model_resize_sizes):
    img = decode_image("assets/dog.png").unsqueeze(0)
    img = preprocess(img, resize_size)

    path = MODEL_PATH[constructor.__name__]
    runtime = Runtime.get()
    program = runtime.load_program(path)
    output, *_ = program.load_method("forward").execute([img])

    predicted_class = weights.meta["categories"][output.argmax().item()]
    print(f"Predicted class for {constructor.__name__}: {predicted_class}")

[program.cpp:154] InternalConsistency verification requested but not available
[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:167] Number of efficient cores 4


Predicted class for efficientnet_v2_s: golden retriever
